In [33]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ================================
# 配置参数
# ================================
DATA_PATH = 'QQQ_Rolling_Validation/QQQ_rolling_validation_dataset.xlsx'
OUTPUT_DIR = 'QQQ_Feature_RankIC_Analysis'
TARGET_NAME = '目标_未来10d回报率'
STABLE_FEATURES_FILE = 'QQQ_XGBoost_IC_Analysis/stable_features.txt'

# ================================
# 核心函数
# ================================

def load_stable_features(file_path=STABLE_FEATURES_FILE):
    """加载稳定特征列表"""
    with open(file_path, 'r', encoding='utf-8') as f:
        features = [line.strip() for line in f if line.strip()]
    print(f"加载了 {len(features)} 个稳定特征")
    return features

def get_available_folds(data_path):
    """获取所有可用的fold编号"""
    excel_file = pd.ExcelFile(data_path)
    fold_sheets = [sheet for sheet in excel_file.sheet_names if sheet.startswith('Fold') and sheet.endswith('_测试集')]
    fold_numbers = [int(sheet.split('Fold')[1].split('_')[0]) for sheet in fold_sheets]
    return sorted(fold_numbers)

def set_chinese_font():
    """设置中文字体"""
    try:
        plt.rcParams['font.sans-serif'] = ['SimHei']
        plt.rcParams['axes.unicode_minus'] = False
    except:
        pass

def analyze_feature_target_rankic_by_fold(data_path, stable_features, target_name):
    """1. 每个fold上特征和目标的RankIC"""
    print("\n1. 每个fold上特征和目标的RankIC")
    print("=" * 60)
    
    available_folds = get_available_folds(data_path)
    all_results = []
    
    for fold_number in available_folds:
        test_sheet = f'Fold{fold_number:02d}_测试集'
        test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
        test_df = test_df.dropna()
        
        print(f"\nFold {fold_number:02d} (样本数: {len(test_df)}):")
        print(f"{'特征名':<40} {'RankIC':<8} {'P值':<8}")
        print("-" * 58)
        
        for feature in stable_features:
            if feature in test_df.columns:
                ic, p_val = stats.spearmanr(test_df[feature], test_df[target_name])
                sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
                print(f"{feature[:39]:<40} {ic:>7.4f}{sig:<3} {p_val:>7.4f}")
                
                all_results.append({
                    'Fold': fold_number,
                    'Feature': feature,
                    'RankIC': round(ic, 4),
                    'P_Value': round(p_val, 4)
                })
    
    return pd.DataFrame(all_results)

def analyze_feature_feature_rankic_by_fold(data_path, stable_features):
    """2. 每个fold上特征和特征之间的RankIC"""
    print("\n\n2. 每个fold上特征和特征之间的RankIC")
    print("=" * 60)
    
    # 确保输出目录存在
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    available_folds = get_available_folds(data_path)
    all_corr_results = []
    
    for fold_number in available_folds:
        test_sheet = f'Fold{fold_number:02d}_测试集'
        test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
        test_df = test_df.dropna()
        
        # 只保留存在的特征
        existing_features = [f for f in stable_features if f in test_df.columns]
        
        # 计算相关矩阵
        corr_matrix = test_df[existing_features].corr(method='spearman').round(4)
        
        print(f"\nFold {fold_number:02d} 特征间相关性矩阵:")
        print(f"特征数量: {len(existing_features)}")
        
        # 找出高相关特征对 (|相关系数| > 0.7)
        high_corr_pairs = []
        for i in range(len(existing_features)):
            for j in range(i+1, len(existing_features)):
                feature1 = existing_features[i]
                feature2 = existing_features[j]
                corr = corr_matrix.loc[feature1, feature2]
                if abs(corr) > 0.7:
                    high_corr_pairs.append({
                        'Fold': fold_number,
                        'Feature1': feature1,
                        'Feature2': feature2,
                        'Correlation': corr
                    })
        
        if high_corr_pairs:
            print(f"高相关特征对 (|相关系数| > 0.7): {len(high_corr_pairs)} 对")
            for pair in high_corr_pairs[:5]:  # 只显示前5对
                print(f"  {pair['Feature1'][:20]:<20} <-> {pair['Feature2'][:20]:<20} : {pair['Correlation']:>7.4f}")
        else:
            print("无高相关特征对 (|相关系数| > 0.7)")
        
        all_corr_results.extend(high_corr_pairs)
        
        # 保存每个fold的相关矩阵
        corr_matrix.to_csv(os.path.join(OUTPUT_DIR, f'fold_{fold_number:02d}_feature_correlation.csv'), 
                          encoding='utf-8-sig')
    
    return pd.DataFrame(all_corr_results)

def analyze_combined_feature_target_rankic(data_path, stable_features, target_name):
    """3. 拼接后测试集上特征和目标的RankIC"""
    print("\n\n3. 拼接后测试集上特征和目标的RankIC")
    print("=" * 60)
    
    available_folds = get_available_folds(data_path)
    test_dfs = []
    
    for fold_number in available_folds:
        test_sheet = f'Fold{fold_number:02d}_测试集'
        test_df = pd.read_excel(data_path, sheet_name=test_sheet, index_col=0, parse_dates=True)
        test_df = test_df.dropna()
        test_dfs.append(test_df)
    
    combined_test_df = pd.concat(test_dfs, axis=0).sort_index()
    print(f"拼接后测试集样本数: {len(combined_test_df)}")
    
    print(f"\n{'特征名':<40} {'RankIC':<8} {'P值':<8}")
    print("-" * 58)
    
    combined_results = []
    for feature in stable_features:
        if feature in combined_test_df.columns:
            ic, p_val = stats.spearmanr(combined_test_df[feature], combined_test_df[target_name])
            sig = "***" if p_val < 0.01 else "**" if p_val < 0.05 else "*" if p_val < 0.1 else ""
            print(f"{feature[:39]:<40} {ic:>7.4f}{sig:<3} {p_val:>7.4f}")
            
            combined_results.append({
                'Feature': feature,
                'RankIC': round(ic, 4),
                'P_Value': round(p_val, 4)
            })
    
    return pd.DataFrame(combined_results), combined_test_df

def analyze_combined_feature_feature_rankic(combined_test_df, stable_features):
    """4. 拼接后测试集上特征和特征之间的RankIC"""
    print("\n\n4. 拼接后测试集上特征和特征之间的RankIC")
    print("=" * 60)
    
    # 只保留存在的特征
    existing_features = [f for f in stable_features if f in combined_test_df.columns]
    
    # 计算相关矩阵
    corr_matrix = combined_test_df[existing_features].corr(method='spearman').round(4)
    
    print(f"特征数量: {len(existing_features)}")
    
    # 找出高相关特征对 (|相关系数| > 0.7)
    high_corr_pairs = []
    for i in range(len(existing_features)):
        for j in range(i+1, len(existing_features)):
            feature1 = existing_features[i]
            feature2 = existing_features[j]
            corr = corr_matrix.loc[feature1, feature2]
            if abs(corr) > 0.7:
                high_corr_pairs.append({
                    'Feature1': feature1,
                    'Feature2': feature2,
                    'Correlation': corr
                })
    
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
    
    print(f"高相关特征对 (|相关系数| > 0.7): {len(high_corr_df)} 对")
    if len(high_corr_df) > 0:
        print("所有高相关特征对:")
        for i, row in high_corr_df.iterrows():
            print(f"  {row['Feature1'][:25]:<25} <-> {row['Feature2'][:25]:<25} : {row['Correlation']:>7.4f}")
    
    return corr_matrix, high_corr_df

def create_heatmap(corr_matrix, output_dir, filename):
    """生成相关性热力图"""
    set_chinese_font()
    
    fig_size = max(8, len(corr_matrix) * 0.4)
    plt.figure(figsize=(fig_size, fig_size))
    
    sns.heatmap(corr_matrix, 
                annot=True,
                fmt='.2f',
                cmap='RdBu_r',
                center=0,
                vmin=-1, vmax=1,
                square=True,
                cbar_kws={'label': 'RankIC'})
    
    plt.title('特征间RankIC相关性热力图', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches='tight')
    plt.close()

def save_results(fold_feature_target, fold_feature_feature, combined_feature_target, 
                combined_corr_matrix, combined_high_corr, output_dir):
    """保存所有结果"""
    os.makedirs(output_dir, exist_ok=True)
    
    # 保存各个分析结果
    fold_feature_target.to_csv(os.path.join(output_dir, '1_fold_feature_target_rankic.csv'), 
                              index=False, encoding='utf-8-sig')
    
    fold_feature_feature.to_csv(os.path.join(output_dir, '2_fold_feature_feature_rankic.csv'), 
                               index=False, encoding='utf-8-sig')
    
    combined_feature_target.to_csv(os.path.join(output_dir, '3_combined_feature_target_rankic.csv'), 
                                  index=False, encoding='utf-8-sig')
    
    combined_corr_matrix.to_csv(os.path.join(output_dir, '4_combined_feature_correlation_matrix.csv'), 
                               encoding='utf-8-sig')
    
    combined_high_corr.to_csv(os.path.join(output_dir, '5_combined_high_correlation_pairs.csv'), 
                             index=False, encoding='utf-8-sig')
    
    # 生成热力图
    create_heatmap(combined_corr_matrix, output_dir, 'combined_feature_correlation_heatmap.png')
    
    print(f"\n结果已保存至: {output_dir}")
    print("文件列表:")
    print("- 1_fold_feature_target_rankic.csv: 各fold特征与目标RankIC")
    print("- 2_fold_feature_feature_rankic.csv: 各fold特征间高相关对")
    print("- 3_combined_feature_target_rankic.csv: 拼接测试集特征与目标RankIC") 
    print("- 4_combined_feature_correlation_matrix.csv: 拼接测试集特征相关矩阵")
    print("- 5_combined_high_correlation_pairs.csv: 拼接测试集高相关特征对")
    print("- combined_feature_correlation_heatmap.png: 特征相关性热力图")
    print("- fold_XX_feature_correlation.csv: 各fold特征相关矩阵")

def main():
    """主函数"""
    if not os.path.exists(DATA_PATH):
        print(f"错误：数据文件不存在 {DATA_PATH}")
        return
    
    if not os.path.exists(STABLE_FEATURES_FILE):
        print(f"错误：稳定特征文件不存在 {STABLE_FEATURES_FILE}")
        return
    
    print("稳定特征RankIC直接分析")
    print("=" * 60)
    
    # 加载稳定特征
    stable_features = load_stable_features(STABLE_FEATURES_FILE)
    
    # 1. 每个fold上特征和目标的RankIC
    fold_feature_target = analyze_feature_target_rankic_by_fold(DATA_PATH, stable_features, TARGET_NAME)
    
    # 2. 每个fold上特征和特征之间的RankIC
    fold_feature_feature = analyze_feature_feature_rankic_by_fold(DATA_PATH, stable_features)
    
    # 3. 拼接后测试集上特征和目标的RankIC
    combined_feature_target, combined_test_df = analyze_combined_feature_target_rankic(DATA_PATH, stable_features, TARGET_NAME)
    
    # 4. 拼接后测试集上特征和特征之间的RankIC
    combined_corr_matrix, combined_high_corr = analyze_combined_feature_feature_rankic(combined_test_df, stable_features)
    
    # 保存所有结果
    save_results(fold_feature_target, fold_feature_feature, combined_feature_target, 
                combined_corr_matrix, combined_high_corr, OUTPUT_DIR)
    
    print(f"\n分析完成！")

if __name__ == "__main__":
    main()

稳定特征RankIC直接分析
加载了 21 个稳定特征

1. 每个fold上特征和目标的RankIC

Fold 01 (样本数: 252):
特征名                                      RankIC   P值      
----------------------------------------------------------
100d峰度                                    0.0336     0.5960
350d偏度                                   -0.0528     0.4041
150d峰度                                   -0.1044*    0.0981
100d夏普比率                                 -0.4672***  0.0000
300d波动率                                   0.4003***  0.0000
300d收益率                                  -0.2854***  0.0000
30d波动率                                    0.3368***  0.0000
300d峰度                                   -0.1403**   0.0260
500d偏度                                    0.0098     0.8772
30d平均收益                                  -0.2351***  0.0002
30d夏普比率                                  -0.2581***  0.0000
100d波动率                                   0.3577***  0.0000
400d峰度                                   -0.1213*    0.0545
50d收益率                       